In [6]:
pip install scipy

  Using cached scipy-1.17.1-cp313-cp313-win_amd64.whl.metadata (60 kB)
Using cached scipy-1.17.1-cp313-cp313-win_amd64.whl (36.5 MB)



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
import pandas as pd
import numpy as np
import os
import timeit

def load_and_clean_data(path):
    if not os.path.exists(path):
        return None
    # Зчитуємо датасет (роздільник крапка з комою)
    p_df = pd.read_csv(path, sep=';', low_memory=False)
    cols = ['Global_active_power', 'Global_reactive_power', 'Voltage', 'Global_intensity', 'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']
    for c in cols: 
        p_df[c] = pd.to_numeric(p_df[c], errors='coerce')
    # Видаляємо всі рядки з пропусками (NaN)
    return p_df.dropna()

# --- Окремі функції вибірок під вимоги лабораторної ---

def get_active_power_high(df):
    # 1. Активна потужність > 5 кВт
    return df[df['Global_active_power'] > 5]

def get_intensity_filter(df):
    # 2. Струм 19-20 А, пралка+холодильник (Sub 1+2) > бойлер+кондіш (Sub 3)
    filt = df[(df['Global_intensity'] >= 19) & (df['Global_intensity'] <= 20)]
    return filt[(filt['Sub_metering_1'] + filt['Sub_metering_2']) > filt['Sub_metering_3']]

def get_random_sample_mean(df):
    # 3. Випадкові 500 000 записів (без повторів) та їх середнє для 3 груп
    sample_size = min(500000, len(df))
    res_sample = df.sample(n=sample_size, replace=False)
    return res_sample[['Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']].mean()

def get_complex_evening_filter(df):
    # 4. Після 18-00, > 6 кВт, група 2 найбільша, крокові зрізи 1-ї та 2-ї половини
    df_copy = df.copy()
    df_copy['Hour'] = df_copy['Time'].apply(lambda x: int(x.split(':')[0]))
    filt = df_copy[(df_copy['Hour'] >= 18) & (df_copy['Global_active_power'] > 6)]
    filt = filt[(filt['Sub_metering_2'] > filt['Sub_metering_1']) & (filt['Sub_metering_2'] > filt['Sub_metering_3'])]
    
    half = len(filt) // 2
    first_half = filt.iloc[:half:3]   # Кожен третій з першої половини
    second_half = filt.iloc[half::4]  # Кожен четвертий з другої половини
    return pd.concat([first_half, second_half])

def normalize_and_standardize(df):
    # 5. Нормалізація (Min-Max) та Стандартизація (Z-score)
    res = df[['Global_active_power', 'Voltage']].copy()
    res['Normalized_Power'] = (res['Global_active_power'] - res['Global_active_power'].min()) / (res['Global_active_power'].max() - res['Global_active_power'].min())
    res['Standardized_Voltage'] = (res['Voltage'] - res['Voltage'].mean()) / res['Voltage'].std()
    return res

def calculate_correlations(df):
    # 6. Коефіцієнти кореляції Пірсона та Спірмена
    p_corr = df['Global_active_power'].corr(df['Global_intensity'], method='pearson')
    s_corr = df['Global_active_power'].corr(df['Global_intensity'], method='spearman')
    return p_corr, s_corr

def run_one_hot_encoding(df):
    # 7. One Hot Encoding категоріального атрибута (створюємо ознаку Day/Night)
    df_sub = df.head(5000).copy()
    df_sub['Hour'] = df_sub['Time'].apply(lambda x: int(x.split(':')[0]))
    df_sub['Day_Type'] = np.where((df_sub['Hour'] >= 6) & (df_sub['Hour'] <= 18), 'Day', 'Night')
    return pd.get_dummies(df_sub['Day_Type'], prefix='Type', dtype=int)

# --- Виконання програми та вимір часу (Профілювання) ---
print("\n" + "="*60)
print("ЛАБОРАТОРНА РОБОТА №2 — ЧАСТИНА 2 (ОНОВЛЕНА)")
print("="*60)

path = 'household_power_consumption.txt'
df_power = load_and_clean_data(path)

if df_power is not None:
    # Завдання 1
    t1 = timeit.timeit(lambda: get_active_power_high(df_power), number=1)
    print(f"\n[Пункт 1] Активна потужність > 5 кВт. Записів: {len(get_active_power_high(df_power))}")
    print(f" > Час виконання (timeit): {t1:.6f} сек.")
    
    # Завдання 2
    t2 = timeit.timeit(lambda: get_intensity_filter(df_power), number=1)
    print(f"\n[Пункт 2] Струм 19-20 А та лічильники. Записів: {len(get_intensity_filter(df_power))}")
    print(f" > Час виконання (timeit): {t2:.6f} сек.")
    
    # Завдання 3
    t3 = timeit.timeit(lambda: get_random_sample_mean(df_power), number=1)
    print(f"\n[Пункт 3] Середнє для 500 000 випадкових записів:")
    print(get_random_sample_mean(df_power))
    print(f" > Час виконання (timeit): {t3:.6f} сек.")
    
    # Завдання 4
    t4 = timeit.timeit(lambda: get_complex_evening_filter(df_power), number=1)
    print(f"\n[Пункт 4] Складна фільтрація після 18-00. Рядків після зрізів: {len(get_complex_evening_filter(df_power))}")
    print(f" > Час виконання (timeit): {t4:.6f} сек.")
    
    # Завдання 5
    print(f"\n[Пункт 5] Нормалізація та стандартизація (Перші 2 рядки):")
    print(normalize_and_standardize(df_power).head(2))
    
    # Завдання 6
    p, s = calculate_correlations(df_power)
    print(f"\n[Пункт 6] Кореляція між потужністю та силою струму:")
    print(f" > Пірсона: {p:.5f} | Спірмена: {s:.5f}")
    
    # Завдання 7
    print(f"\n[Пункт 7] One Hot Encoding для типу часу (Перші 2 рядки):")
    print(run_one_hot_encoding(df_power).head(2))
    
else:
    print(f"\n[!] Файл '{path}' не знайдено.")
    print("Будь ласка, завантажте датасет і покладіть його в папку з проєктом.")

print("\n" + "="*60)
print(" РОБОТУ ЗАВЕРШЕНО УСПІШНО")
print("="*60)


ЛАБОРАТОРНА РОБОТА №2 — ЧАСТИНА 2 (ОНОВЛЕНА)

[Пункт 1] Активна потужність > 5 кВт. Записів: 17547
 > Час виконання (timeit): 0.021153 сек.

[Пункт 2] Струм 19-20 А та лічильники. Записів: 5579
 > Час виконання (timeit): 0.035376 сек.

[Пункт 3] Середнє для 500 000 випадкових записів:
Sub_metering_1    1.135460
Sub_metering_2    1.308548
Sub_metering_3    6.469234
dtype: float64
 > Час виконання (timeit): 0.352420 сек.

[Пункт 4] Складна фільтрація після 18-00. Рядків після зрізів: 310
 > Час виконання (timeit): 3.022291 сек.

[Пункт 5] Нормалізація та стандартизація (Перші 2 рядки):
   Global_active_power  Voltage  Normalized_Power  Standardized_Voltage
0                4.216   234.84          0.374796             -1.851816
1                5.360   233.63          0.478363             -2.225274

[Пункт 6] Кореляція між потужністю та силою струму:
 > Пірсона: 0.99889 | Спірмена: 0.99537

[Пункт 7] One Hot Encoding для типу часу (Перші 2 рядки):
   Type_Day  Type_Night
0         1     